In [14]:
import sympy as sp

q1, q2, q3 = sp.symbols('q1 q2 q3', real=True)
dq1, dq2, dq3 = sp.symbols('dq1 dq2 dq3', real=True)
ddq1, ddq2, ddq3 = sp.symbols('ddq1 ddq2 ddq3', real=True)

m1, m2 = sp.symbols('m1 m2', real=True, positive=True)
L1 = sp.symbols('L1', real=True, positive=True) # 第一连杆长度
d1, d2 = sp.symbols('d1 d2', real=True, positive=True) # 质心距离

# 转动惯量
I1xx, I1yy, I1zz = sp.symbols('I1xx I1yy I1zz', real=True, positive=True)
I2xx, I2yy, I2zz = sp.symbols('I2xx I2yy I2zz', real=True, positive=True)

# 动态等效重力向量
gx, gy, gz = sp.symbols('gx gy gz', real=True)
g_vec = sp.Matrix([gx, gy, gz])

# 状态向量
q = sp.Matrix([q1, q2, q3])
dq = sp.Matrix([dq1, dq2, dq3])

# 基坐标系X前，Y左，Z上
# q1绕Z轴旋转
R_z = sp.Matrix([
    [sp.cos(q1), -sp.sin(q1), 0],
    [sp.sin(q1),  sp.cos(q1), 0],
    [0,           0,          1]
])

# 规定q为正时，机械臂抬起
def rot_y_neg(th):
    return sp.Matrix([
        [sp.cos(th), 0, -sp.sin(th)],
        [0,          1,  0         ],
        [sp.sin(th), 0,  sp.cos(th)]
    ])

R_p1 = rot_y_neg(q2)
R_p2 = rot_y_neg(q3)
R_p12 = rot_y_neg(q2 + q3)

R1 = R_z * R_p1
R2 = R_z * R_p12

# 质心位置
# p_joint12 = [0,0,0]
p_c1 = R1 * sp.Matrix([d1, 0, 0])
p_joint3 = R1 * sp.Matrix([L1, 0, 0])
p_c2 = p_joint3 + R2 * sp.Matrix([d2, 0, 0])

# 平动求导
v_c1 = p_c1.jacobian(q) * dq
v_c2 = p_c2.jacobian(q) * dq

# 角速度
omega1_global = sp.Matrix([0, 0, dq1]) + R_z * sp.Matrix([0, -dq2, 0])
omega2_global = sp.Matrix([0, 0, dq1]) + R_z * sp.Matrix([0, -dq2 - dq3, 0])
omega1_local = R1.T * omega1_global     # 将角速度转换到各自连杆的局部坐标系下，以使用对角化的惯量矩阵
omega2_local = R2.T * omega2_global

I1 = sp.diag(I1xx, I1yy, I1zz)
I2 = sp.diag(I2xx, I2yy, I2zz)

# 总动能
T_trans = 0.5 * m1 * v_c1.dot(v_c1) + 0.5 * m2 * v_c2.dot(v_c2)
T_rot = 0.5 * (omega1_local.T * I1 * omega1_local)[0] + 0.5 * (omega2_local.T * I2 * omega2_local)[0]
T = sp.simplify(T_trans + T_rot)

# 总势能
V = - m1 * g_vec.dot(p_c1) - m2 * g_vec.dot(p_c2)
V = sp.simplify(V)

M = sp.zeros(3, 3)
for i in range(3):
    for j in range(3):
        M[i, j] = sp.diff(sp.diff(T, dq[i]), dq[j])
M = sp.simplify(M)

C = sp.zeros(3, 3)
for k in range(3):
    for j in range(3):
        c_val = 0
        for i in range(3):
            # c_ijk = 1/2 * ( dM_kj/dqi + dM_ki/dqj - dM_ij/dqk )
            c_ijk = 0.5 * (sp.diff(M[k, j], q[i]) + sp.diff(M[k, i], q[j]) - sp.diff(M[i, j], q[k]))
            c_val += c_ijk * dq[i]
        C[k, j] = c_val
C = sp.simplify(C)

G = sp.zeros(3, 1)
for i in range(3):
    G[i] = sp.diff(V, q[i])
G = sp.simplify(G)

print("\nM:")
print(M)
# sp.pprint(M)

print("\nC:")
sp.pprint(C)

print("\ng(q):")
sp.pprint(G)

for i in range(G.rows):
  print(sp.ccode(G[i, 0], assign_to=f"G[{i}]"))


M:
Matrix([[1.0*I1xx*sin(q2)**2 + 1.0*I1zz*cos(q2)**2 + 1.0*I2xx*sin(q2 + q3)**2 + 1.0*I2zz*cos(q2 + q3)**2 + 1.0*d1**2*m1*cos(q2)**2 + 1.0*m2*(L1**2*cos(q2)**2 + 2*L1*d2*cos(q2)*cos(q2 + q3) - 2*d2**2*sin(q2)*sin(q3)*cos(q2 + q3) + d2**2*cos(q2)**2 + d2**2*cos(q3)**2 - d2**2), 0, 0], [0, 1.0*I1yy + 1.0*I2yy + 1.0*d1**2*m1 + 1.0*m2*(L1**2 + 2*L1*d2*cos(q3) + d2**2), 1.0*I2yy + 1.0*d2*m2*(L1*cos(q3) + d2)], [0, 1.0*I2yy + 1.0*d2*m2*(L1*cos(q3) + d2), 1.0*I2yy + 1.0*d2**2*m2]])

C:
⎡          ⎛                                                                   ↪
⎢- 0.5⋅dq₂⋅⎝-I1xx⋅sin(2⋅q₂) + I1zz⋅sin(2⋅q₂) - I2xx⋅sin(2⋅q₂ + 2⋅q₃) + I2zz⋅si ↪
⎢                                                                              ↪
⎢                                                                      ⎛       ↪
⎢                                                              0.5⋅dq₁⋅⎝-I1xx⋅ ↪
⎢                                                                              ↪
⎣                           

In [12]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

# 设需要辨识的组合参数为 b1 = m1*d1 + m2*L1, b2 = m2*d2
b1, b2 = sp.symbols('b1 b2', real=True)
theta_sym = sp.Matrix([b1, b2])

Y_sym = sp.zeros(3, 2)
Y_sym[:, 0] = sp.simplify(sp.diff(G, d1) / m1)
Y_sym[:, 1] = sp.simplify(sp.diff(G, d2) / m2)

print("\n参数向量theta:")
sp.pprint(theta_sym.T)
print("\n回归矩阵Y:")
sp.pprint(Y_sym)

error = sp.simplify((Y_sym * sp.Matrix([m1*d1 + m2*L1, m2*d2])) - G)
print(f"\n{error.is_zero_matrix}")

Y_func = sp.lambdify((q1, q2, q3, gx, gy, gz), Y_sym, "numpy")

def gravity_regressor_3d(q_val, g_val):
    res = np.array(Y_func(*q_val, *g_val), dtype=float)
    return res.reshape(3, 2)

def identify_gravity_params_3d(q_data, g_data, tau_g_data):
    N = len(q_data)
    Y_stack = np.vstack([gravity_regressor_3d(q_data[i], g_data[i]) for i in range(N)])
    tau_stack = tau_g_data.reshape(-1)
    
    reg = LinearRegression(fit_intercept=False)
    reg.fit(Y_stack, tau_stack)
    
    theta_hat = reg.coef_
    score = reg.score(Y_stack, tau_stack)
    return theta_hat, score


df = pd.read_csv("data_20260510_041540.csv")
print(f"data size: {len(df)}")

q1_arr = df["q1"].values
q2_arr = df["q2"].values
q3_arr = df["q3"].values
q_data = np.column_stack((q1_arr, q2_arr, q3_arr))

tau_g_data = df[["tau1", "tau2", "tau3"]].values

g0 = 9.8 
g_data = np.zeros((len(df), 3))
g_data[:, 2] = -g0

# 执行辨识
theta_hat, r2 = identify_gravity_params_3d(q_data, g_data, tau_g_data)
print(f"b1 = {theta_hat[0]:.4f},  b2 = {theta_hat[1]:.4f}")
print(f"R^2:   {r2:.6f}")


参数向量theta:
[b₁  b₂]

回归矩阵Y:
⎡         (gx⋅sin(q₁) - gy⋅cos(q₁))⋅cos(q₂)                          (gx⋅sin(q ↪
⎢                                                                              ↪
⎢gx⋅sin(q₂)⋅cos(q₁) + gy⋅sin(q₁)⋅sin(q₂) - gz⋅cos(q₂)  gx⋅sin(q₂ + q₃)⋅cos(q₁) ↪
⎢                                                                              ↪
⎣                         0                            gx⋅sin(q₂ + q₃)⋅cos(q₁) ↪

↪ ₁) - gy⋅cos(q₁))⋅cos(q₂ + q₃)               ⎤
↪                                             ⎥
↪  + gy⋅sin(q₁)⋅sin(q₂ + q₃) - gz⋅cos(q₂ + q₃)⎥
↪                                             ⎥
↪  + gy⋅sin(q₁)⋅sin(q₂ + q₃) - gz⋅cos(q₂ + q₃)⎦

True
data size: 45
b1 = 0.5665,  b2 = 0.2648
R^2:   0.993206


In [13]:
tau_g_sym = Y_sym * sp.Matrix([b1, b2])

for i in range(3):
    # 将符号转为 C 代码
    c_code = sp.ccode(sp.simplify(tau_g_sym[i]), assign_to=f"tau_g[{i}]")
    print(c_code)


tau_g[0] = (b1*cos(q2) + b2*cos(q2 + q3))*(gx*sin(q1) - gy*cos(q1));
tau_g[1] = b1*(gx*sin(q2)*cos(q1) + gy*sin(q1)*sin(q2) - gz*cos(q2)) + b2*(gx*sin(q2 + q3)*cos(q1) + gy*sin(q1)*sin(q2 + q3) - gz*cos(q2 + q3));
tau_g[2] = b2*(gx*sin(q2 + q3)*cos(q1) + gy*sin(q1)*sin(q2 + q3) - gz*cos(q2 + q3));
